# Files & Exceptions

This notebook introduces file handling in Python.

## Learning Objectives

By the end of this notebook, you should be able to:

- Read a text file into a string or a list of lines, and process it one line at a time.
- Use a file context manager to close a file automatically.
- Write or append text, choosing the appropriate file mode and line breaks.
- Explain how `try` and `except` handle exceptions, and identify relevant exception types.
- Read and write tabular data with the `csv` module, including headers, delimiters, and numeric conversion.

## Setup

You will need to run the following cell to copy the necessary files into a local data directory:

In [ ]:
# Use the bundled data locally; download missing files when running in Colab.
from pathlib import Path
from urllib.request import urlretrieve

base_url = "https://raw.githubusercontent.com/olearydj/INSY3010/main/lectures/14a-files-and-exceptions/data"
Path("data").mkdir(exist_ok=True)
for name in ["the_file.txt", "auburn_creed.txt", "zillow.csv", "zillow-tabs.csv",
             "word_count_example.txt", "word_count_empty.txt"]:
    destination = Path("data") / name
    if not destination.exists():
        urlretrieve(f"{base_url}/{name}", destination)
        print(f"Downloaded {name}")

## Reading Files

So far, input has come from the keyboard via `input()`, but we'll often want to read data in from a file. There are two main types of files:

- text files: human readable (txt, csv, html, json, log, etc.)
- binary files: not human readable (exe, zip, jpg, etc.)

Text files range from completely unstructured to very structured. Base  Python includes tools suitable for both. We will cover plain text (e.g. TXT) and CSV formats. The methods we describe can be applied to any non-binary format, though specific packages are available for most structured formats (e.g. JSON).

### Plain Text Formats

The general steps are to open the file, read from it, and close the file. You may choose to process data while reading from it, or after closing the file.

### Open, Read, and Close

To open a file, use the aptly named `open` function. `open()` takes a filename or path, along with optional arguments that control how the file is opened, such as `mode` and `encoding`. The file name may include a relative or absolute path. Relative paths are resolved from the current working directory, which may differ from the directory containing a script or notebook. Open returns a *file object* that is used to "handle" the data.

Text files store characters using an **encoding**, a rule for translating between characters and bytes. **UTF-8** is a widely used encoding that supports text from many languages. In this course, use `encoding="utf-8"` when opening text files for reading or writing, unless the file is known to use a different encoding. Using the same encoding when saving and reading avoids relying on system defaults.

In [ ]:
file_name = './data/the_file.txt'
file_handle = open(file_name, encoding="utf-8")
print(file_handle)

Here, `file_handle` is the name of the object used to manipulate `the_file.txt`. File objects like `file_handle` have several methods, including:

- `.read()` - returns the full file contents as a single string
- `.readlines()` - returns a list of strings, one for each line in the file
- `.readline()` - returns a single line at a time
- `.close()` - closes the file object

When reading a text file, each line normally includes its ending newline (`\n`). The final line may have no newline. A blank line is represented by `"\n"`, while `readline()` returns `""` when there is nothing left to read.

The following code reads the data associated with the file and closes the handle before printing the contents.

In [ ]:
contents = file_handle.read()
file_handle.close()
print(contents)

### Contents as a List of Strings

It is often useful to read in the content as a list of strings, where each string represents a line in the file. This is accomplished with the `readlines` method.

In [ ]:
file_name = './data/the_file.txt'
file_handle = open(file_name, encoding="utf-8")

lines = file_handle.readlines()
file_handle.close()

print(lines)

This provides direct access to individual lines via indexing or groups via slicing.

In [ ]:
# first line
print(lines[0])

# list of all lines, in reverse order
print(lines[::-1])

### One Line at a Time

Alternatively, you can read lines one at a time.

This is typically used when you want to process each line as it is read, and avoid loading the whole file first. Why might you want to do that?

In [ ]:
file_name = './data/the_file.txt'
file_handle = open(file_name, encoding="utf-8")
file_handle.readline()

Note the trailing newline on this output. We've not used `print`, so the cell output is simply the last value **returned** by its contents, a string (denoted by the single quotes), including the trailing newline character.

We haven't closed the file, so more readline operations can be performed:

In [ ]:
file_handle.readline()

In [ ]:
file_handle.readline()

Once we reach the end of a file, `readline` will always return an empty string.

### Looping through Lines

Reading line by line is typically done in a loop. We can use the empty string returned by `readline` at the end of a file to trigger the end of a `while` loop.

In [ ]:
# close previous handle
file_handle.close()

file_name = './data/the_file.txt'
file_handle = open(file_name, encoding="utf-8")

# loop through all lines
while True:
    line = file_handle.readline()
    if line == '':
        break
    print(line)

# close it again
file_handle.close()

Note the blank line introduced by this approach. Where does it come from? How would you correct for it?

However, file handle objects are iterable, so it is more common to use a `for` loop. This approach avoids the need for an explicit `readline`, making it simple and easy to read.

In [ ]:
file_name = './data/the_file.txt'
file_handle = open(file_name, encoding="utf-8")

for line in file_handle:
    print(line, end='')

file_handle.close()

### Exercise - Word Counts

Write `count_words(filename)` to read a text file and return a dictionary whose keys are words and whose values are their occurrence counts.

Use whitespace to separate words. Preserve capitalization and punctuation: `"Blue"`, `"blue"`, and `"blue,"` count as different words. An empty file should return `{}`.

The supplied `data/word_count_example.txt` contains:

```text
red blue red
blue green
```

The supplied `data/word_count_empty.txt` is empty.

In [ ]:
# Define count_words(filename).

Run these checks after implementing your function.

In [ ]:
assert count_words("data/word_count_example.txt") == {
    "red": 2, "blue": 2, "green": 1
}
assert count_words("data/word_count_empty.txt") == {}
print(count_words("data/auburn_creed.txt"))

#### Solution / Discussion

In [ ]:
def count_words(fn):
    fh = open(fn, encoding="utf-8")
    d = {}
    for line in fh:
        for word in line.split():
            if word in d:
                d[word] += 1
            else:
                d[word] = 1
    fh.close()
    return d

How would you modify this code to ignore "stop words" (commonly used words that convey little meaning, e.g. “the”, “a”, “an”, or “in”) and eliminate punctuation at the end of words?

## Discussion

- Structure in text files varies
  - Impacts how you process resulting data
  - Line breaks can be used
    - to separate lines or paragraphs
    - inconsistently
    - not at all
  - Tools exist to help with this, e.g. `textwrap.wrap()`
- Interacting with file systems
  - Specify a **path** relative to the current working directory, or use an absolute path
  - Path names are different between Windows and MacOS / Linux
    - Windows: `c:\dev\insy3010`
    - MacOS / Linux: `~/dev/insy3010`
  - To write Python that works in both, use `pathlib`

## Context Managers

In Python, a **context manager** can be used to handle *setup and teardown* operations associated with a task. When working with files, a context manager will automatically close the handle when access is complete. This is accomplished by the `with ... as` syntax.

In [ ]:
file_name = './data/the_file.txt'

with open(file_name, encoding="utf-8") as file_handle:
    lines = file_handle.readlines()

In [ ]:
for idx, line in enumerate(lines):
    print(f'{idx:02d}: {line.strip()}')

Confirm that the file is automatically closed:

In [ ]:
# no close required
line = file_handle.readline()

It is good practice to always use `with` when reading files.

Context managers are available in a variety of contexts, but their cleanup behavior depends on the object. A file context manager closes the file. For SQLite connections, continue using the explicit `commit()` and `close()` pattern from 13b.

## Error Handling

It is the programmer's responsibility to prevent and/or handle all errors that may occur. When working with files this is particuarly important to avoid large scale data loss or corruption.

In some cases the most effective way to do this is to let them happen and "catch" them.

Contrasting approaches:

- anticipate and correct errors (look before you leap, lbyl)
- accept they may happen and catch them (ask forgiveness not permission, afnp)

Here is an example of the former:

In [ ]:
# Check if the input is valid BEFORE trying to convert it
def lbyl_approach(user_input):
    '''check if input is numeric before converting, warn if not'''
    if user_input.isdigit() or (user_input.startswith('-') and user_input[1:].isdigit()):
        number = int(user_input)
        print(f"LBYL: Successfully converted '{user_input}' to {number}")
        return number
    else:
        print(f"LBYL: '{user_input}' is not a valid integer")
        return None

test_inputs = ["42", "-17", "abc", "12.5", "  "]

print("=== LBYL Approach ===")
for inp in test_inputs:
    lbyl_approach(inp)

Works but relatively complex and fragile.

To implement the second option, Python provides the `try ... except` mechanism:

```python
try:
    # if any error occurs in this code block...
except:
    # this code runs
```

Here is the same problem implemented with try / except:

In [ ]:
# Try to convert it and handle the exception if it fails
def afnp_approach(user_input):
    '''try to convert and warn if it fails'''
    try:
        number = int(user_input)
        print(f"AFNP: Successfully converted '{user_input}' to {number}")
        return number
    except ValueError:
        print(f"AFNP: '{user_input}' is not a valid integer")
        return None


test_inputs = ["42", "-17", "abc", "12.5", "  "]

print("\n=== AFNP Approach ===")
for inp in test_inputs:
    afnp_approach(inp)

This is a simple and straightforward approach that is very robust.

Try / except can be extended to process different error types differently, for example, the success of the `open` statement depends on the existence of the specified file, among other things. The following code sample uses a specific except block to catch `FileNotFoundErrors` and another to catch anything else that might go wrong:


```python
try:
    with open("missing.txt", encoding="utf-8") as f:
        ...  # do stuff with f
except FileNotFoundError:
    print("File not found!")
    # handle this gracefully - get different filename?
except:
    # handle any other error
    print("Other error in file open")
    raise  # allows the error to crash the program
```

## Writing Files

Data can be written to an open file using the `write` method, which writes strings. But the file handle must have write permissions, which are set when it is created.

```text
>>> help(open)

Help on function open in module _io:

open(file, mode='r', buffering=-1, encoding=None, errors=None, newline=None, closefd=True, opener=None)
    Open file and return a stream.  Raise OSError upon failure.
```

A variety of modes exist, but we will focus on these:

- mode 'r' is the default, allows read only
- mode 'w' gives write permissions, will create a new file or **overwrite** existing
- mode 'a' opens for appending, will create a new file or add to an existing one

The general steps are to open the file in the appropriate mode, write to the file, and close it. You can either prepare the data to write before those steps or as part of the write process.

When opening a file for the `csv` module, also use `newline=""`. This lets the module handle line endings itself and avoids extra blank lines on some systems, especially Windows. We will use this argument in the CSV examples below.

In [ ]:
with open("./data/write_test.txt", "a", encoding="utf-8") as f:
    f.write("File opened for append.")
    for idx in range(5):
        f.write(f"Writing line {idx}")
    f.write("Closing file.")

The resulting file looks like this:

```text
File opened for append.Writing line 0Writing line 1Writing line 2Writing line 3Writing line 4Closing file.
```

Why is everything on one line?

### Exercise - Append a Line

Write `add_line(filename, line)` to append a string followed by a newline to a UTF-8 text file, using a file context manager. The caller supplies text without a trailing newline. Assume an existing nonempty file already ends with a newline; create the file if it does not exist.

Preserve existing contents. Print a success message and return `True` after a successful write. If an error occurs, print an error message and return `False`.

In [ ]:
# Define add_line(filename, line).

Run these checks after implementing your function. It resets its small test file each time so the expected contents stay the same.

In [ ]:
with open("data/append_example.txt", "w", encoding="utf-8") as f:
    f.write("First line\n")

assert add_line("data/append_example.txt", "Second line") is True
with open("data/append_example.txt", encoding="utf-8") as f:
    assert f.read() == "First line\nSecond line\n"

# A directory cannot be opened as a text file for appending.
assert add_line("data", "Cannot write this") is False

#### Solution / Discussion

In [ ]:
def add_line(filename, line):
    try:
        with open(filename, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except:
        print("Could not append line.")
        return False
    print("Line added successfully.")
    return True

The function adds the newline, so callers do not need to supply it. Appending preserves the existing text; it does not automatically insert a separator before the new text. That is why the contract assumes an existing nonempty file already ends with a newline.

An integer cannot be combined with the newline string. The function catches the error, prints its error message, and returns `False`.

In [ ]:
assert add_line("data/append_example.txt", 42) is False

## CSV Files

Comma Separated Values (CSV) files are use a structured, plain text format. They provide a simple way to represent tabular data. Each line in a CSV represents a row in the table, with the value of each column in a comma separated list.

For example, here part of a CSV of housing data from Zillow (spaces added for clarity):

```text
 1, 2222, 3, 3.5, 32312, 1981, 250000
 2, 1628, 3, 2,   32308, 2009, 185000
 3, 3824, 5, 4,   32312, 1954, 399000
 4, 1137, 3, 2,   32309, 1993, 150000
 5, 3560, 6, 4,   32309, 1973, 315000
 ```

Here, each line has values corresponding to the line number (index), square footage, bedrooms, bathrooms, zipcode, year built, and list price.

The CSV format is very common, especially for export / import. It is not usually created or edited by hand, but generated, e.g. by Excel. There are better formats for large or complex datasets, but CSV is often *good enough*.

Python provides a `csv` module for manipulating this file type.

### Reading CSV

Import the module, create a file handle, wrap it in a `csv.reader` object. The resulting interface is an iterable that yields each row of the CSV as a list of strings. Each item in the list is a field from the row.

For example, reading the first row of the Zillow data:

`1,2222,3,3.5,32312,1981,250000`

will return:

`['1','2222','3','3.5','32312','1981','250000']`

Note that each element is a string.

In [ ]:
import csv

with open("./data/zillow.csv", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

Sometime CSV files will include a header line with the column names. This can be read separately using the `next` function, which returns the next value in an iterable.

Variants of CSV exist where tab or other characters are used to separate fields. To handle those, `csv.reader` has a `delimiter` parameter that specifies the single character separating fields—for example, `","` for commas or `"\t"` for tabs.

For example, consider this tab separated version of the Zillow file that includes a header line:

```text
Index	Sq Ft	Beds	Baths	Zip	Year	List Price ($)
1	2222	3	3.5	32312	1981	250000
2	1628	3	2	32308	2009	185000
3	3824	5	4	32312	1954	399000
4	1137	3	2	32309	1993	150000
5	3560	6	4	32309	1973	315000
```

In [ ]:
import csv
# function for pretty printing data structures
from pprint import pprint as pp

with open("./data/zillow-tabs.csv", encoding="utf-8", newline="") as f:
    # specify tab-delimited format
    reader = csv.reader(f, delimiter='\t')

    # build a list of rows, starting with the header
    zillow = []
    zillow.append(next(reader))
    for row in reader:
        zillow.append(row)

pp(zillow)

### Exercise - Type Conversion

CSV fields arrive as strings. Write `convert_type(row)` to try converting each value to an integer. If that fails, convert it to a float instead. Use `try / except` inside a loop.

Return a **new list** and leave the input row unchanged. For this demonstration, treat every field as numeric and assume each value can be converted to an integer or a float. Do not pass the header to this function. Then modify the CSV-reading example above to convert each data row while retaining the header.

In [ ]:
# Define convert_type(row), then use it while reading the Zillow data.

Run these checks after implementing your function.

In [ ]:
sample_row = ["1", "2222", "3", "3.5", "32312", "1981", "250000"]
original_row = sample_row.copy()
converted = convert_type(sample_row)
assert converted == [1, 2222, 3, 3.5, 32312, 1981, 250000]
assert [type(value) for value in converted] == [int, int, int, float, int, int, int]
assert sample_row == original_row
assert converted is not sample_row

#### Solution / Discussion

**Failed attempt: convert every field to an integer.** This deliberately raises `ValueError` at the decimal bathroom count. We need a fallback when integer conversion fails.

In [ ]:
def convert_all_to_int(row):
    return [int(value) for value in row]

convert_all_to_int(["1", "2222", "3", "3.5", "32312", "1981", "250000"])

**Working solution: try integers, then floats.** Each iteration tries `int(value)`. If that raises an exception, the handler appends `float(value)` instead.

In [ ]:
import csv
from pprint import pprint as pp

def convert_type(row):
    converted = []
    for value in row:
        try:
            converted.append(int(value))
        except:
            converted.append(float(value))
    return converted

with open("data/zillow-tabs.csv", encoding="utf-8", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    zillow = [next(reader)]
    for row in reader:
        zillow.append(convert_type(row))

pp(zillow)

This example treats every field as numeric to practice `try / except`. In real data, ZIP codes are identifiers and should usually remain strings so leading zeros are preserved. Also, a value such as `"3"` becomes an integer while `"3.5"` becomes a float; the resulting type follows the text. If a value cannot be converted to either type, the float conversion will still raise an error.

**Alternative: a helper function and a list comprehension.** Put the conversion of one value in a helper function, then use the list-comprehension pattern from Python + SQL II to build the row. This produces the same result as the loop above.

In [ ]:
def convert_value(value):
    try:
        return int(value)
    except:
        return float(value)

def convert_type(row):
    return [convert_value(value) for value in row]

### Writing CSV Files

The `csv.writer()` object is used to write CSV formatted data to a file handle with appropriate permissions. There are two relevant methods:

- `writerow(row_data)` writes a single row, where `row_data` is a list of values
- `writerows(table_data)` writes multiple rows, where `table_data` is a list of lists of values

Everything is written in plain text, without any information about the underlying data types. This information will have to be reconstructed when read into Python, Excel, etc.

In [ ]:
import csv

zillow_data = [
    [1, 2222, 3, 3.5, 32312, 1981, 250000],
    [2, 1628, 3, 2, 32308, 2009, 185000],
    [3, 3824, 5, 4, 32312, 1954, 399000],
    [4, 1137, 3, 2, 32309, 1993, 150000],
    [5, 3560, 6, 4, 32309, 1973, 315000]
]

header = ['Index', 'Sq Ft', 'Beds', 'Baths', 'Zip', 'Year', 'List Price ($)']

with open('./data/zillow-new.csv', 'w', encoding='utf-8', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(header)
    writer.writerows(zillow_data)

print("Export complete.")

## Best Practices

- Use `with open(...) as f:` to close files automatically, including when an exception interrupts the block.
- Choose the file mode deliberately: `"r"` reads, `"w"` replaces existing contents, and `"a"` adds to the end. Check the destination before writing.
- Use explicit `encoding="utf-8"` for text files you control. For CSV files, also use `newline=""` and let the `csv` module handle separators and quoting.
- Iterate over a file when you can process one line at a time; read all its contents when you need to reuse or index them.
- Catch the specific exceptions you can handle, such as `FileNotFoundError`, `OSError`, or `ValueError`. Keep the `try` block focused on the operation that may fail.
- Verify written data by opening the output and comparing its contents with what you intended to save.

## Common Gotchas

- Relative paths start from the current working directory. A file beside your notebook may still be missing from the directory where the code runs.
- Reading advances the file's position. At the end of a file, `readline()` returns `""`; reopen the file when you need a fresh pass.
- A blank line is usually `"\n"`, not `""`. The last line of a text file may have no trailing newline.
- `print(line)` adds its own newline. If `line` already ends in `"\n"`, use `print(line, end="")` to avoid an extra blank line.
- `write()` does not add line breaks. Include `"\n"` where you want a new line, and remember that rerunning append code adds the text again.
- A file is closed after its `with` block ends. Process it inside the block, or save the data for later use.
- CSV fields are read as strings, even when they look numeric. Convert fields according to their meaning; identifiers such as ZIP codes may need to remain strings to preserve leading zeros.
- A context manager closes a file but does not undo writes. An exception handler also does not guarantee that an output file is complete or correct.

## Glossary

| Term | Meaning |
|---|---|
| Path | A location in a file system, expressed relative to a working directory or as an absolute location |
| Working directory | The directory used as the starting point for relative paths |
| File object / handle | The object returned by `open()`, used to read, write, and close a file |
| File mode | An option selecting how a file is opened, such as reading, writing, or appending |
| Newline | A character marking a line break, represented in Python strings as `\n` |
| End of file (EOF) | The point at which no unread content remains in a file |
| Context manager | An object used with `with` to manage setup and cleanup; a file context manager closes the file when the block exits |
| Exception | An error reported during execution that can interrupt the normal flow of a program |
| Exception handler | An `except` block that runs when a matching exception occurs in its associated `try` block |
| CSV | Comma-separated values: a text format for tabular data, with quoting rules for fields containing special characters |
| Delimiter | The character separating fields in a record, such as a comma or tab |
| Header | A row naming the columns in a table or data file |
| Encoding | The rule used to translate between text characters and stored bytes, such as UTF-8 |

## Practice Problems

Solve these independently. Use UTF-8 and file context managers. Run each problem's checks after implementing your function. Solutions are kept separately for the instructor.

### ★ 1. Count Lines

Write `count_lines(filename)` to return the number of lines in a text file, including blank lines. An empty file has zero lines. A final line still counts when it has no trailing newline. Assume the file exists and can be read.

In [ ]:
# Define count_lines(filename).

Run the following cell to test your work.

In [ ]:
with open("data/practice_lines.txt", "w", encoding="utf-8") as f:
    f.write("first\n\nlast")
with open("data/practice_empty.txt", "w", encoding="utf-8") as f:
    f.write("")

assert count_lines("data/practice_lines.txt") == 3
assert count_lines("data/practice_empty.txt") == 0

### ★★ 2. Read Numbers

Write `read_numbers(filename)` for a text file containing one value per line. Try converting each line to a float. Use `try / except` to skip lines that cannot be converted, including blank lines. Return `(numbers, skipped_count)`, where `numbers` is a list of floats in file order. Assume the file exists and can be read; keep file-access errors separate from handling invalid values.

In [ ]:
# Define read_numbers(filename).

Run the following cell to test your work.

In [ ]:
with open("data/practice_numbers.txt", "w", encoding="utf-8") as f:
    f.write("12\ninvalid\n\n-3.5\n 4 \n")
with open("data/practice_empty.txt", "w", encoding="utf-8") as f:
    f.write("")

numbers, skipped = read_numbers("data/practice_numbers.txt")
assert numbers == [12.0, -3.5, 4.0]
assert all(type(value) is float for value in numbers)
assert skipped == 2
assert read_numbers("data/practice_empty.txt") == ([], 0)

### ★★ 3. Fix This Code: A Closed File

The following function should return the nonblank lines of a file with surrounding whitespace removed. Instead, it tries to read a closed file. Explain why, then write a corrected `read_nonblank(filename)`. Assume the file exists and can be read.

```python
def read_nonblank(filename):
    with open(filename, encoding="utf-8") as f:
        lines = []
    for line in f:
        if line.strip():
            lines.append(line.strip())
    return lines
```

In [ ]:
# Define the corrected read_nonblank(filename).

Run the following cell to test your work.

In [ ]:
with open("data/practice_nonblank.txt", "w", encoding="utf-8") as f:
    f.write("  alpha  \n\n   \n beta\n")
with open("data/practice_empty.txt", "w", encoding="utf-8") as f:
    f.write("")

assert read_nonblank("data/practice_nonblank.txt") == ["alpha", "beta"]
assert read_nonblank("data/practice_empty.txt") == []

### ★★ 4. Export a Report to CSV

Write `write_report(filename, headers, rows)` using the `csv` module, UTF-8, `newline=""`, and a file context manager. Replace any existing output file. Write the header followed by the supplied rows, in order. Empty data should produce a header-only file. No return value is required; assume the destination is writable.

Use the checks to read back the result and verify the header, row count, and a text field containing a comma. Explain why manually joining values with commas would not handle that field correctly.

In [ ]:
# Define write_report(filename, headers, rows).

Run the following cell to test your work.

In [ ]:
import csv

headers = ["Location", "Count"]
rows = [["North, Annex", 3], ["South", 0]]
write_report("data/practice_report.csv", headers, rows)
with open("data/practice_report.csv", encoding="utf-8", newline="") as f:
    saved = list(csv.reader(f))
assert saved[0] == headers
assert len(saved) == 3
assert saved[1:] == [["North, Annex", "3"], ["South", "0"]]

write_report("data/practice_report.csv", headers, [])
with open("data/practice_report.csv", encoding="utf-8", newline="") as f:
    assert list(csv.reader(f)) == [headers]

---

Auburn University / Industrial and Systems Engineering  
INSY 3010 / Programming and Databases for ISE / Fall 2026  
© Copyright 2026, Danny J. O'Leary.  
For licensing, attribution, and information: [GitHub INSY3010](https://github.com/olearydj/INSY3010)